<a href="https://colab.research.google.com/github/Josephleme/plp-da-a3-duka-sales-and-agent-cleanup/blob/main/agent_cleanup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mobile Money Agent Network Transaction Cleanup & Exploration


In [2]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(44)
n = 600

agents = pd.DataFrame({
    "agent_id": [f"AG{i:03d}" for i in range(1, 21)],
    "county": rng.choice(["Nairobi", "Kiambu", "Nakuru", "Mombasa", "Kisumu"], 20),
    "onboarded": pd.date_range("2024-01-01", periods=20, freq="MS").strftime("%Y-%m-%d"),
})

txns = pd.DataFrame({
    "txn_id": [f"TX{i:05d}" for i in range(1, n + 1)],
    "agent_id": rng.choice(agents["agent_id"], n),
    "txn_type": rng.choice(["Deposit", "withdrawal", "DEPOSIT ", "Withdrawal", "deposit"], n),
    "amount": rng.lognormal(7.5, 0.8, n).round(0).astype(str),
    "txn_date": (pd.to_datetime("2026-03-01") + pd.to_timedelta(rng.integers(0, 60, n), unit="D")).strftime("%Y-%m-%d"),
})

idx = rng.choice(n, 30, replace=False)
txns.loc[idx[:15], "amount"] = np.nan
txns.loc[idx[15:20], "amount"] = "N/A"
txns.loc[idx[20:25], "amount"] = "250000"
txns.loc[idx[25:], "txn_date"] = txns.loc[idx[25:], "txn_date"].str.replace("-", "/")
txns = pd.concat([txns, txns.sample(12, random_state=1)], ignore_index=True)
txns.loc[rng.choice(len(txns), 8, replace=False), "agent_id"] = "AG099"

agents.to_csv("agents.csv", index=False)
txns.to_csv("agent_transactions.csv", index=False)


In [3]:
import pandas as pd

raw_txns = pd.read_csv("agent_transactions.csv", keep_default_na=False)
raw_agents = pd.read_csv("agents.csv")

print(f"Total Rows: {len(raw_txns)}")
print(f"Duplicate Rows: {raw_txns.duplicated().sum()}")
print(f"Inconsistent Types: {len(raw_txns[~raw_txns['txn_type'].isin(['Deposit', 'Withdrawal'])])}")
print(f"Slash Dates: {raw_txns['txn_date'].str.contains('/').sum()}")
print(f"Missing/NA Amounts: {((raw_txns['amount'] == '') | (raw_txns['amount'] == 'N/A') | (raw_txns['amount'] == 'nan')).sum()}")
print(f"Outlier Entries: {(raw_txns['amount'] == '250000').sum()}")
print(f"Unknown Agents: {(raw_txns['agent_id'] == 'AG099').sum()}")


Total Rows: 612
Duplicate Rows: 11
Inconsistent Types: 384
Slash Dates: 5
Missing/NA Amounts: 22
Outlier Entries: 5
Unknown Agents: 8


In [4]:
import pandas as pd

audit_data = {
    "issue": [
        "Duplicate transactions from system sync lag",
        "Mixed casing and trailing whitespace variations",
        "Non-standard date formats using slash delimiters",
        "Missing values and structural N/A placeholder strings",
        "Extreme anomalous value injections",
        "Transactions referencing unregistered identification codes"
    ],
    "dimension": [
        "Uniqueness",
        "Consistency",
        "Validity",
        "Completeness",
        "Accuracy",
        "Validity"
    ],
    "column": [
        "Multiple",
        "txn_type",
        "txn_date",
        "amount",
        "amount",
        "agent_id"
    ],
    "rows affected": [11, 384, 5, 22, 5, 8]
}

df_audit = pd.DataFrame(audit_data)
df_audit


,issue,dimension,column,rows affected
0,Duplicate transactions from system sync lag,Uniqueness,Multiple,11
1,Mixed casing and trailing whitespace variations,Consistency,txn_type,384
2,Non-standard date formats using slash delimiters,Validity,txn_date,5
3,Missing values and structural N/A placeholder ...,Completeness,amount,22
4,Extreme anomalous value injections,Accuracy,amount,5
5,Transactions referencing unregistered identifi...,Validity,agent_id,8


In [6]:
import os
from cleaning_pipeline import clean_agent_data

os.makedirs("outputs", exist_ok=True)

txns_raw = pd.read_csv("agent_transactions.csv")
agents_raw = pd.read_csv("agents.csv")

clean_df, execution_log = clean_agent_data(txns_raw, agents_raw)

print(f"Pre-cleaned dataset rows: {len(txns_raw)}")
print(f"Post-cleaned validated rows: {len(clean_df)}")
print(f"Total duplicate rows removed: {execution_log['duplicates_removed']}")
print(f"Total missing values imputed: {execution_log['missing_amounts_imputed']}")
print(f"Transactions with unknown agents: {execution_log['unknown_agent_transactions']}")

clean_df.to_csv("outputs/agent_transactions_clean.csv", index=False)


Pre-cleaned dataset rows: 612
Post-cleaned validated rows: 601
Total duplicate rows removed: 11
Total missing values imputed: 20
Transactions with unknown agents: 8


In [7]:
import numpy as np

county_summary = clean_df.groupby(["county", "txn_type"]).agg(
    total_value=("amount", "sum"),
    transaction_count=("txn_id", "count"),
    average_value=("amount", "mean")
)
print("--- COUNTY-LEVEL TRANSACTION SUMMARY ---")
print(county_summary)

clean_df["txn_date"] = pd.to_datetime(clean_df["txn_date"])
clean_df["week"] = clean_df["txn_date"].dt.isocalendar().week

weekly_pivot = clean_df.pivot_table(
    values="amount",
    index="week",
    columns="county",
    aggfunc="sum",
    fill_value=0
)
print("\n--- WEEKLY TRANSACTION VALUE BY COUNTY PIVOT ---")
print(weekly_pivot)


--- COUNTY-LEVEL TRANSACTION SUMMARY ---
                    total_value  transaction_count  average_value
county  txn_type                                                 
Kiambu  Deposit        334737.0                 43    7784.581395
        Withdrawal      77024.5                 27    2852.759259
Kisumu  Deposit        436521.0                 74    5898.932432
        Withdrawal      76348.5                 37    2063.472973
Mombasa Deposit        149120.0                 72    2071.111111
        Withdrawal     368592.0                 61    6042.491803
Nairobi Deposit        543724.0                118    4607.830508
        Withdrawal     161932.0                 62    2611.806452
Nakuru  Deposit        184957.0                 60    3082.616667
        Withdrawal     328568.5                 39    8424.833333

--- WEEKLY TRANSACTION VALUE BY COUNTY PIVOT ---
county    Kiambu    Kisumu   Mombasa   Nairobi    Nakuru
week                                                    
9  

### Data Quality Report

* **Uniqueness:** The system audit detected 11 duplicate records caused by legacy database synchronization lag. These were successfully dropped programmatically to establish a single source of truth without loss of valid operations.
* **Consistency:** The `txn_type` column contained 384 inconsistent records due to mixed-case strings and trailing spaces. These were standardized into uniform 'Deposit' and 'Withdrawal' groups using programmatic text normalization.
* **Validity:** Date formats containing slash delimiters were normalized into standard ISO-8601 formatting before being parsed as datetime objects. In addition, 8 transactions referencing the unknown agent code 'AG099' were safely preserved but flagged as missing county context.
* **Completeness:** The dataset contained 22 missing or 'N/A' placeholder strings in the amount column. Of these, 20 were successfully imputed using the historical median value grouped by transaction type, and an tracking flag was appended to mark these adjustments.
* **Accuracy:** Statistical profiling using the standard Interquartile Range (IQR) method flagged extreme transaction amounts. This step successfully isolated the KES 250,000 value injections as outliers to protect downstream mathematical calculations from skew.
* **Timeliness:** All evaluated transaction records fall within the expected 60-day historical logging window spanning March and April 2026, satisfying corporate operational reporting requirements.

### Regulatory Compliance & Privacy Framework

If this transaction extract contained customer phone numbers, they would be classified as personally identifiable information (PII) under the Kenya Data Protection Act 2019. To protect this sensitive data, a SHA-256 cryptographic hashing pipeline would be implemented to mask individual phone numbers into irreversible tokens before exporting files. Furthermore, strict role-based access control (RBAC) policies must be enforced, ensuring that raw operational datasets remain encrypted at rest using AES-256 keys, with decryption rights restricted solely to authorized compliance officers.
